In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model ="claude-sonnet-4-5"

In [ ]:
def add_user_message(messages, text):
    user_message = ({"role": "user", "content": text})
    messages.append(user_message)

def add_assistant_message(messages, text):
    assitant_message = ({"role": "assistant", "content": text})
    messages.append(assitant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return "".join(block.text for block in response.content if block.type == "text")

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result."""
    prompt = f"""
Please solve the following task:
{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "``` code")
    output = chat(messages, stop_sequences=["```"])
    return output

## Implementing a Model Grader

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 weaknesses
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string[],
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages,"```json")
    eval_text = chat(messages, stop_sequences = ["```"])
    return json.loads(eval_text)

# Integrating Grading into Your Workflow

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

def run_eval(dataset):
    """Loads the dataset and call run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

# Calculate an average score across all test cases:

In [ ]:
from statistics import mean

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results


In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))